In [23]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import geopandas as gpd
from datetime import datetime
import json
import re
import time
from shapely.geometry import shape
from pathlib import Path

GEOJSON_OUTPUT_FP = Path("output/espoo_feedback.geojson")

# Load
gdf = gpd.read_file(GEOJSON_OUTPUT_FP)

# Remove duplicate rows
gdf = gdf.drop_duplicates(subset="id")

# Save back to the same file
# gdf.to_file(GEOJSON_OUTPUT_FP, driver="GeoJSON")

c:\Users\fonta\repos\geospatial_challenge_camp\venv\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Several features with id = 383588 have been found. Altering it to be unique. This warning will not be emitted anymore for this layer
  return ogr_read(


In [24]:
gdf.to_file(GEOJSON_OUTPUT_FP, driver="GeoJSON")
# gdf = gdf.reset_index()

In [25]:
gdf.to_file(GPKG_OUTPUT_FP, layer="feedback", driver="GPKG")

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import geopandas as gpd
from datetime import datetime
import json
import re
import time
from shapely.geometry import shape
from pathlib import Path

GPKG_OUTPUT_FP = Path("output/espoo_feedback.gpkg")
GEOJSON_OUTPUT_FP = Path("output/espoo_feedback.geojson")
LAYER = "feedback"

base_url = "https://easiointi.espoo.fi/eFeedback/en/View/{}"
start_id = 379075
N = 2000

headers = {
    "User-Agent": "Mozilla/5.0"
}


# -----------------------------
# Load existing dataset
# -----------------------------
if GPKG_OUTPUT_FP.exists():
    existing_gdf = gpd.read_file(GPKG_OUTPUT_FP, layer=LAYER)
    existing_ids = set(existing_gdf["id"])
    print(existing_ids)
    print(f"Loaded {len(existing_gdf)} existing records")
else:
    existing_gdf = None
    existing_ids = set()

start_id = min([*existing_ids, start_id + 1]) - 1
records = []

for i in range(N):
    feedback_id = start_id - i

    if feedback_id in existing_ids:
        continue
    
    print(f"Getting feedback {feedback_id}")

    url = base_url.format(feedback_id)

    r = requests.get(url, headers=headers)

    if r.status_code != 200:
        # print(f"Skipping {feedback_id} (status {r.status_code})")
        continue

    soup = BeautifulSoup(r.text, "html.parser")

    # feedback text
    feedback_div = soup.find("div", class_="view-propertyvalue")

    if not feedback_div:
        continue

    feedback_text = feedback_div.get_text(strip=True)

    if not feedback_text.strip():
        # print(f"Skipping {feedback_id} (no feedback text)")
        continue


    # creation time
    creation_time = None

    for prop in soup.find_all("span", class_="view-propertyname"):
        if "Creation time" in prop.get_text():
            value = prop.find_next_sibling("span", class_="view-propertyvalue")
            if value:
                creation_time = value.get_text(strip=True)

                # convert to datetime
                creation_time = datetime.strptime(creation_time, "%d.%m.%Y")


    # category
    category_tag = soup.find("span", class_="feedback-title black")

    if category_tag:
        category = category_tag.get_text(strip=True)
    else:
        category = None

    # --- geometry info from EmbeddedIMSAddGeometry(...) ---
    streetaddress = None
    geometry_dict = None
    geometry_type = None
    geometry_obj = None
    for script in soup.find_all("script"):
        script_text = script.string or script.get_text()
        if "EmbeddedIMSAddGeometry(" not in script_text:
            continue

        match = re.search(
            r'EmbeddedIMSAddGeometry\((\[.*?\])\);',
            script_text,
            re.DOTALL
        )
        if not match:
            continue

        try:
            items = json.loads(match.group(1))
            if items:
                item = items[0]
                geometry_dict = item.get("geometry")
                streetaddress = item.get("streetaddress")

                if geometry_dict:
                    geometry_type = geometry_dict.get("type")
                    geometry_obj = shape(geometry_dict)  # handles Point, LineString, Polygon, etc.
        except json.JSONDecodeError:
            pass

        break


    records.append({
        "id": feedback_id,
        "creation_time": creation_time,
        "category": category,
        "feedback": feedback_text,
        "streetaddress": streetaddress,
        "geometry_type": geometry_type,
        "geometry_raw": geometry_dict,
        "geometry": geometry_obj,
        "url": url
    })

    print(f"Collected feedback {feedback_id}")

    df = pd.DataFrame(records)
    gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:3067")


    gdf_combined = gpd.GeoDataFrame(
        pd.concat([existing_gdf, gdf], ignore_index=True),
        crs=gdf.crs
    )

    gdf_combined.to_file(
        GPKG_OUTPUT_FP,
        layer=LAYER,
        driver="GPKG",
        mode='w'
    )
    gdf_combined.to_file("output/espoo_feedback.geojson", driver="GeoJSON")

    print(f"Saved {feedback_id}")

    # time.sleep(1)  # polite delay


{382468, 382982, 384019, 383523, 384038, 381482, 383032, 385093, 382032, 383569, 381524, 384599, 384091, 385116, 379998, 382561, 384612, 383588, 382054, 381542, 380006, 380012, 384109, 385647, 384111, 381551, 385650, 381557, 384630, 381567, 382603, 383119, 380052, 383125, 380053, 385690, 384672, 383140, 383143, 382119, 384681, 384685, 382637, 380085, 384698, 382651, 382139, 381628, 382145, 384719, 384731, 384735, 382178, 385251, 384739, 382179, 381670, 385260, 384237, 385262, 382190, 384240, 384767, 383234, 385286, 384266, 383250, 381205, 383766, 382235, 384800, 384807, 382255, 383796, 384827, 382268, 383809, 381249, 384328, 382281, 381259, 381263, 382290, 384858, 384348, 383837, 384352, 381794, 384868, 384869, 383336, 381800, 381805, 381293, 384374, 384375, 383350, 384396, 382868, 381336, 381851, 381347, 381860, 384937, 381866, 383409, 383410, 382897, 384437, 383927, 381890, 383427, 384456, 381902, 385490, 383959, 381403, 381916, 383965, 384992, 382946, 381926, 383975, 383980, 383477,

ConnectionError: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))